# Data Mining Lab Assignment 2: Medallion Data Lake & Recommender System

**Student Name:** Prashik Humane  
**Roll Number:** LCB2023039  
**Course:** Data Mining  
**Dataset:** Book-Crossing (Users, Books, Book-Ratings)  
**Environment:** Google Colab / Python 3  

---

## Executive Summary & Architecture Overview

This project implements a production-grade **Medallion Data Lake Architecture** (**Bronze $\to$ Silver $\to$ Gold**) to ingest, clean, aggregate, and transform the raw, highly sparse Book-Crossing dataset into model-ready assets for an **Item-Based Collaborative Filtering Recommender System**.

```
+-----------------------------------------------------------------------------------+
|                            MEDALLION DATA LAKE ARCHITECTURE                       |
+-----------------------------------------------------------------------------------+
|                                                                                   |
|  [Raw CSVs]                                                                       |
|  BX-Users.csv                                                                     |
|  BX-Books.csv        ===>   BRONZE LAYER (Untouched Raw Storage)                  |
|  BX-Ratings.csv             * Semicolon-delimited, Latin-1 encoded                |
|                             * Appended UTC Ingestion Timestamp                    |
|                             * Parquet Format: users, books, ratings               |
|                                     |                                             |
|                                     v                                             |
|                             EXPLORATORY DATA ANALYSIS (EDA)                       |
|                             * Missing values, rating zero-spike (62.28%)          |
|                             * Age/Year anomalies, long-tail power law             |
|                             * Matrix Sparsity: 99.9968%                           |
|                                     |                                             |
|                                     v                                             |
|                             SILVER LAYER (Cleaned & Filtered)                     |
|                             * Exact deduplication (ISBN, User-ID)                 |
|                             * Kept explicit ratings (1-10), dropped 0s            |
|                             * Age imputation (median = 32), Location split        |
|                             * Validated publication years, cleaned text           |
|                             * Foreign key alignment & Interaction filter (>= 10)  |
|                                     |                                             |
|                                     v                                             |
|                             GOLD LAYER (Model-Ready Assets)                       |
|                             * Book-User Sparse Rating Matrix (CSR)                |
|                             * Book Metadata Lookup Table                          |
|                             * User & Book Aggregates (count, avg, min, max)       |
|                                     |                                             |
|                                     v                                             |
|                    ITEM-BASED COLLABORATIVE FILTERING RECOMMENDER                 |
|                    * Cosine Similarity via NearestNeighbors                      |
|                    * Title similarity search: get_similar_books(title, n)         |
|                    * User Recommender: recommend(user_id, n)                      |
|                    * Rating-weighted ranking with read-item exclusion             |
|                                                                                   |
+-----------------------------------------------------------------------------------+
```


## Setup & Data Ingestion Preparation
In this step, we ensure directory structures (`bronze/`, `silver/`, `gold/`) exist and download the Book-Crossing dataset if not already present in the runtime environment.


In [1]:
import os
import sys
import zipfile
import urllib.request
import datetime
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict

# Create Medallion Lake Folders
for folder in ['bronze', 'silver', 'gold']:
    os.makedirs(folder, exist_ok=True)
    print(f"Directory ready: {folder}/")

# Download dataset if not present
dataset_url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
zip_filename = "book-crossings.zip"

if not os.path.exists(zip_filename):
    print("Downloading Book-Crossing dataset...")
    urllib.request.urlretrieve(dataset_url, zip_filename)
    print("Download complete!")
else:
    print(f"Dataset '{zip_filename}' already exists locally.")

# Extract files
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(".")
    extracted_files = zip_ref.namelist()
    print("Extracted files:", extracted_files)


Directory ready: bronze/
Directory ready: silver/
Directory ready: gold/
Dataset 'book-crossings.zip' already exists locally.
Extracted files: ['BX-Book-Ratings.csv', 'BX-Books.csv', 'BX-Users.csv']


---
## Step 1: Set Up the Data Lake and Build the Bronze Layer

**Bronze Layer Principles:**
- The Bronze layer is an **exact, immutable, untouched copy** of raw source data.
- The raw Book-Crossing files use **semicolon (`;`) separators** and **non-UTF-8 (`latin-1`) encoding**, with occasional unescaped quotes handled via `on_bad_lines='skip'`.
- We append an `ingestion_timestamp` to track pipeline lineage and store each table in Apache **Parquet** format for columnar compression and fast I/O.


In [1]:
# Ingest raw source files without cleaning
ingestion_ts = datetime.datetime.now(datetime.timezone.utc).isoformat()

# 1. Users table
df_users_raw = pd.read_csv('BX-Users.csv', sep=';', encoding='latin-1', on_bad_lines='skip', low_memory=False)
df_users_raw['ingestion_timestamp'] = ingestion_ts
df_users_raw.to_parquet('bronze/users_bronze.parquet', index=False)

# 2. Books table
df_books_raw = pd.read_csv('BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip', low_memory=False)
df_books_raw['ingestion_timestamp'] = ingestion_ts
df_books_raw.to_parquet('bronze/books_bronze.parquet', index=False)

# 3. Ratings table
df_ratings_raw = pd.read_csv('BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip', low_memory=False)
df_ratings_raw['ingestion_timestamp'] = ingestion_ts
df_ratings_raw.to_parquet('bronze/ratings_bronze.parquet', index=False)

print(f"[Bronze] Users Table Ingested   : Shape = {df_users_raw.shape}")
print(f"[Bronze] Books Table Ingested   : Shape = {df_books_raw.shape}")
print(f"[Bronze] Ratings Table Ingested : Shape = {df_ratings_raw.shape}")
print(f"[Bronze] Ingestion Timestamp    : {ingestion_ts}")


[Bronze] Users Table Ingested   : Shape = (278858, 4)
[Bronze] Books Table Ingested   : Shape = (271360, 9)
[Bronze] Ratings Table Ingested : Shape = (1149780, 4)
[Bronze] Ingestion Timestamp    : 2026-11-23T05:16:11.834125+00:00


---
## Step 2: Explore the Bronze Data (EDA)

We load the Bronze Parquet tables back into memory and perform exploratory data analysis across all 5 required dimensions.
For each item, we report our exact quantitative findings and the corresponding remediation action to be taken in the **Silver layer**.


### 2.1 Shape, Data Types, and Missing-Value Count


In [1]:
# Read Bronze Parquet tables
bronze_users = pd.read_parquet('bronze/users_bronze.parquet')
bronze_books = pd.read_parquet('bronze/books_bronze.parquet')
bronze_ratings = pd.read_parquet('bronze/ratings_bronze.parquet')

print("=== USERS TABLE ===")
print("Shape:", bronze_users.shape)
print("Data Types:")
print(bronze_users.dtypes)
print("Missing Values:")
print(bronze_users.isnull().sum())

print("\n=== BOOKS TABLE ===")
print("Shape:", bronze_books.shape)
print("Data Types:")
print(bronze_books.dtypes)
print("Missing Values:")
print(bronze_books.isnull().sum())

print("\n=== RATINGS TABLE ===")
print("Shape:", bronze_ratings.shape)
print("Data Types:")
print(bronze_ratings.dtypes)
print("Missing Values:")
print(bronze_ratings.isnull().sum())


=== USERS TABLE ===
Shape: (278858, 4)
Data Types:
User-ID                  int64
Location                object
Age                    float64
ingestion_timestamp     object
dtype: object
Missing Values:
User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64

=== BOOKS TABLE ===
Shape: (271360, 9)
Data Types:
ISBN                   object
Book-Title             object
Book-Author            object
Year-Of-Publication    object
Publisher              object
Image-URL-S            object
Image-URL-M            object
Image-URL-L            object
ingestion_timestamp    object
dtype: object
Missing Values:
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
ingestion_timestamp    0
dtype: int64

=== RATINGS TABLE ===
Shape: (1149780, 4)
Data Types:
User-ID          

**EDA Finding & Silver Action:**  
> **Finding:** `users` has 278,858 rows with 110,762 missing ages (39.72%), `books` has 271,360 rows with unneeded image URLs and non-numeric year entries, and `ratings` has 1,149,780 rows with zero missing values.  
> **Silver Action:** Drop image URLs, cast types cleanly, impute missing user ages, and remove corrupted/dangling records.


### 2.2 Distribution of Ratings (Implicit vs. Explicit)


In [1]:
rating_counts = bronze_ratings['Book-Rating'].value_counts().sort_index()
zero_ratings = rating_counts.get(0, 0)
explicit_ratings = bronze_ratings[bronze_ratings['Book-Rating'] > 0]['Book-Rating']

print("Rating Distribution:")
for rating, count in rating_counts.items():
    pct = count / len(bronze_ratings) * 100
    print(f"  Rating {rating:2d}: {count:8d} ({pct:5.2f}%)")

pct_zero = zero_ratings / len(bronze_ratings) * 100
pct_explicit = len(explicit_ratings) / len(bronze_ratings) * 100
print(f"\nTotal Zero (Implicit) Ratings : {zero_ratings:,} ({pct_zero:.2f}%)")
print(f"Total Explicit (1-10) Ratings: {len(explicit_ratings):,} ({pct_explicit:.2f}%)")


Rating Distribution:
  Rating  0:   716109 (62.28%)
  Rating  1:     1770 ( 0.15%)
  Rating  2:     2759 ( 0.24%)
  Rating  3:     5996 ( 0.52%)
  Rating  4:     8904 ( 0.77%)
  Rating  5:    50974 ( 4.43%)
  Rating  6:    36924 ( 3.21%)
  Rating  7:    76457 ( 6.65%)
  Rating  8:   103736 ( 9.02%)
  Rating  9:    67541 ( 5.87%)
  Rating 10:    78610 ( 6.84%)

Total Zero (Implicit) Ratings : 716,109 (62.28%)
Total Explicit (1-10) Ratings: 433,671 (37.72%)


**EDA Finding & Silver Action:**  
> **Finding:** 716,109 ratings (62.28%) have a value of 0 denoting implicit interactions rather than explicit user ratings, while 433,671 ratings span the 1–10 scale peaking at 8 and 10.  
> **Silver Action:** Filter out all 0 ratings and retain strictly explicit ratings between 1 and 10 for collaborative filtering.


### 2.3 Distribution of User Ages and Publication Years


In [1]:
# User Age Analysis
ages = bronze_users['Age']
print("User Age Statistics:")
print(f"  Min Age: {ages.min()}, Max Age: {ages.max()}, Mean Age: {ages.mean():.2f}")
print(f"  Users with Age < 5   : {(ages < 5).sum():,}")
print(f"  Users with Age > 100 : {(ages > 100).sum():,}")
valid_ages = ages[(ages >= 5) & (ages <= 100)]
median_age = valid_ages.median()
print(f"  Valid Age Median (5 to 100): {median_age:.1f}")

# Publication Year Analysis
pub_years_numeric = pd.to_numeric(bronze_books['Year-Of-Publication'], errors='coerce')
print("\nPublication Year Anomalies:")
print(f"  Non-numeric years (text shifted): {pub_years_numeric.isna().sum()}")
print(f"  Year == 0                       : {(pub_years_numeric == 0).sum():,}")
print(f"  Future Years (> 2026)           : {(pub_years_numeric > 2026).sum():,}")


User Age Statistics:
  Min Age: 0.0, Max Age: 244.0, Mean Age: 34.75
  Users with Age < 5   : 882
  Users with Age > 100 : 366
  Valid Age Median (5 to 100): 32.0

Publication Year Anomalies:
  Non-numeric years (text shifted): 3
  Year == 0                       : 4,618
  Future Years (> 2026)           : 11


**EDA Finding & Silver Action:**  
> **Finding:** User ages contain biologically impossible outliers (0 to 244 with 882 users < 5 and 366 > 100), and publication years contain 4,618 zero values, 11 future dates (> 2026), and 3 misaligned text strings.  
> **Silver Action:** Set ages < 5 or > 100 to null and impute with median age (32), and set publication years <= 0 or > 2026 to null.


### 2.4 Number of Ratings per User and per Book


In [1]:
exp_ratings_df = bronze_ratings[bronze_ratings['Book-Rating'] > 0]
user_activity = exp_ratings_df['User-ID'].value_counts()
book_popularity = exp_ratings_df['ISBN'].value_counts()

print("Ratings per User Quantiles:")
print(user_activity.quantile([0.5, 0.75, 0.90, 0.95, 0.99]))
u_gte_10 = (user_activity >= 10).sum()
u_total = len(user_activity)
u_pct = u_gte_10 / u_total * 100
print(f"Users with >= 10 ratings: {u_gte_10:,} out of {u_total:,} ({u_pct:.2f}%)")

print("\nRatings per Book Quantiles:")
print(book_popularity.quantile([0.5, 0.75, 0.90, 0.95, 0.99]))
b_gte_10 = (book_popularity >= 10).sum()
b_total = len(book_popularity)
b_pct = b_gte_10 / b_total * 100
print(f"Books with >= 10 ratings: {b_gte_10:,} out of {b_total:,} ({b_pct:.2f}%)")


Ratings per User Quantiles:
0.50     1.0
0.75     3.0
0.90     9.0
0.95    19.0
0.99    73.0
Name: count, dtype: float64
Users with >= 10 ratings: 7,334 out of 77,805 (9.43%)

Ratings per Book Quantiles:
0.50     1.0
0.75     2.0
0.90     4.0
0.95     6.0
0.99    22.0
Name: count, dtype: float64
Books with >= 10 ratings: 5,645 out of 185,973 (3.04%)


**EDA Finding & Silver Action:**  
> **Finding:** User and book interaction distributions follow an extreme power law: over 50% of users and books have only 1 rating, and only 9.43% of users and 3.04% of books have 10 or more ratings.  
> **Silver Action:** Filter the dataset to users and books with at least 10 ratings to eliminate cold-start noise and build reliable cosine similarity vectors.


### 2.5 Matrix Sparsity


In [1]:
n_users_raw = bronze_ratings['User-ID'].nunique()
n_books_raw = bronze_ratings['ISBN'].nunique()
n_ratings_raw = len(bronze_ratings)
possible_pairs = n_users_raw * n_books_raw
sparsity_raw = (1.0 - (n_ratings_raw / possible_pairs)) * 100
density_raw = (n_ratings_raw / possible_pairs) * 100

print(f"Unique Users in Ratings  : {n_users_raw:,}")
print(f"Unique Books in Ratings  : {n_books_raw:,}")
print(f"Total Rating Records     : {n_ratings_raw:,}")
print(f"Total Possible Pairs     : {possible_pairs:,}")
print(f"Raw Matrix Density       : {density_raw:.6f}%")
print(f"Raw Matrix Sparsity      : {sparsity_raw:.6f}%")


Unique Users in Ratings  : 105,283
Unique Books in Ratings  : 340,556
Total Rating Records     : 1,149,780
Total Possible Pairs     : 35,854,757,348
Raw Matrix Density       : 0.003207%
Raw Matrix Sparsity      : 99.996793%


**EDA Finding & Silver Action:**  
> **Finding:** The raw rating matrix is 99.9968% sparse, meaning only 0.0032% of all potential user-book interaction pairs contain an observed rating.  
> **Silver Action:** Prune the long tail in Silver to increase interaction density by two orders of magnitude so collaborative filtering can identify genuine neighborhood overlaps.


---
## Step 3: Clean the Data and Build the Silver Layer

### Transformation Rules Applied:
1. **Deduplication**: Remove duplicate rows across users, books, and user-book rating pairs.
2. **Explicit Ratings**: Retain strictly explicit ratings from 1 to 10 (discarding 0).
3. **Age Imputation**: Replace user ages $< 5$ or $> 100$ with `NaN`, then impute with the median age (32).
4. **Location Splitting**: Deconstruct `Location` into `city`, `state`, and `country` columns in Title Case.
5. **Publication Year Validation**: Convert years to numeric; coerce 0 or future years ($> 2026$) to `NaN`.
6. **Text Cleaning**: Strip whitespace, standardise string cases, uppercase ISBNs, and drop unneeded image URLs (`Image-URL-S`, `Image-URL-M`, `Image-URL-L`).
7. **Foreign Key Integrity**: Retain only ratings where both `User-ID` exists in `users` and `ISBN` exists in `books`.
8. **Interaction Threshold ($\ge 10$)**:
   - **Chosen Threshold:** At least 10 ratings per user AND at least 10 ratings per book.
   - **EDA Justification:** The EDA demonstrated that over 90% of books have fewer than 5 ratings and over 50% have only 1 rating. Unfiltered collaborative filtering on single-interaction items produces degenerate orthogonal similarity vectors ($cosine = 0$) or spurious random noise. A threshold of 10 retains 5,440 popular books and 6,319 active users across 87,233 explicit ratings, raising matrix density from **0.0032% to 0.2538%** (a **79-fold density increase**) while maintaining strong catalog diversity.


In [1]:
# Load bronze tables
users_b = pd.read_parquet('bronze/users_bronze.parquet')
books_b = pd.read_parquet('bronze/books_bronze.parquet')
ratings_b = pd.read_parquet('bronze/ratings_bronze.parquet')

# Standardize IDs and keys before deduplication
users_b['User-ID'] = pd.to_numeric(users_b['User-ID'], errors='coerce')
users_s = users_b[users_b['User-ID'].notna()].copy()
users_s['User-ID'] = users_s['User-ID'].astype(int)

books_b['ISBN'] = books_b['ISBN'].astype(str).str.strip().str.upper()
books_s = books_b.copy()

ratings_b['ISBN'] = ratings_b['ISBN'].astype(str).str.strip().str.upper()
ratings_b['User-ID'] = pd.to_numeric(ratings_b['User-ID'], errors='coerce')
ratings_s = ratings_b[ratings_b['User-ID'].notna()].copy()
ratings_s['User-ID'] = ratings_s['User-ID'].astype(int)

# 1. Deduplicate
users_s = users_s.drop_duplicates(subset=['User-ID']).copy()
books_s = books_s.drop_duplicates(subset=['ISBN']).copy()
ratings_s = ratings_s.drop_duplicates(subset=['User-ID', 'ISBN']).copy()

# 2. Filter ratings 1 to 10
ratings_s = ratings_s[(ratings_s['Book-Rating'] >= 1) & (ratings_s['Book-Rating'] <= 10)].copy()

# 3. Fix user ages: set < 5 or > 100 to NaN, then fill with median (32)
users_s.loc[(users_s['Age'] < 5) | (users_s['Age'] > 100), 'Age'] = np.nan
valid_median_age = users_s.loc[(users_s['Age'] >= 5) & (users_s['Age'] <= 100), 'Age'].median()
users_s['Age'] = users_s['Age'].fillna(valid_median_age).astype(int)

# 4. Split user location into city, state, country
loc_parts = users_s['Location'].astype(str).str.split(',', n=2, expand=True)
users_s['city'] = loc_parts[0].str.strip().str.title().replace('', np.nan)
users_s['state'] = (loc_parts[1].str.strip().str.title().replace('', np.nan) if 1 in loc_parts.columns else np.nan)
users_s['country'] = (loc_parts[2].str.strip().str.title().replace('', np.nan) if 2 in loc_parts.columns else np.nan)

# 5. Fix publication years: 0 or future years to NaN
pub_years = pd.to_numeric(books_s['Year-Of-Publication'], errors='coerce')
books_s['Year-Of-Publication'] = pub_years.apply(lambda y: int(y) if (pd.notna(y) and 1800 <= y <= 2026) else np.nan)

# 6. Clean text columns & drop image URLs
books_s.drop(columns=['Image-URL-S', 'Image-URL-M', 'Image-URL-L'], inplace=True, errors='ignore')
books_s['Book-Title'] = books_s['Book-Title'].astype(str).str.strip()
books_s['Book-Author'] = books_s['Book-Author'].astype(str).str.strip()
books_s['Publisher'] = books_s['Publisher'].astype(str).str.strip()

# 7. Foreign key constraint: book and user must exist in catalog
ratings_s = ratings_s[
    ratings_s['ISBN'].isin(books_s['ISBN']) & 
    ratings_s['User-ID'].isin(users_s['User-ID'])
].copy()

# 8. Minimum rating threshold (>= 10 for both users and books)
min_rating_threshold = 10
user_rating_counts = ratings_s['User-ID'].value_counts()
book_rating_counts = ratings_s['ISBN'].value_counts()

active_users = user_rating_counts[user_rating_counts >= min_rating_threshold].index
active_books = book_rating_counts[book_rating_counts >= min_rating_threshold].index

ratings_silver = ratings_s[
    ratings_s['User-ID'].isin(active_users) & 
    ratings_s['ISBN'].isin(active_books)
].copy()

# Keep corresponding users and books in silver tables
users_silver = users_s[users_s['User-ID'].isin(ratings_silver['User-ID'].unique())].copy()
books_silver = books_s[books_s['ISBN'].isin(ratings_silver['ISBN'].unique())].copy()

# Save cleaned Silver tables to Parquet
users_silver.to_parquet('silver/users_silver.parquet', index=False)
books_silver.to_parquet('silver/books_silver.parquet', index=False)
ratings_silver.to_parquet('silver/ratings_silver.parquet', index=False)

print("=== SILVER LAYER SUMMARY ===")
print(f"Users Silver   : {users_silver.shape} (Age nulls: {users_silver['Age'].isnull().sum()})")
print(f"Books Silver   : {books_silver.shape} (Columns: {list(books_silver.columns)})")
print(f"Ratings Silver : {ratings_silver.shape} (Score range: {ratings_silver['Book-Rating'].min()} to {ratings_silver['Book-Rating'].max()})")
density_silver = len(ratings_silver) / (users_silver.shape[0] * books_silver.shape[0]) * 100
print(f"Silver Matrix Density : {density_silver:.4f}% (Sparsity: {100 - density_silver:.4f}%)")


=== SILVER LAYER SUMMARY ===
Users Silver   : (6319, 7) (Age nulls: 0)
Books Silver   : (5440, 6) (Columns: ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'ingestion_timestamp'])
Ratings Silver : (87233, 4) (Score range: 1 to 10)
Silver Matrix Density : 0.2538% (Sparsity: 99.7462%)


---
## Step 4: Build the Gold Layer

**Gold Layer Assets Created:**
1. **Book-User Sparse Matrix (`csr_matrix`)**: Compressed Sparse Row matrix where each row represents a book, each column represents a user, and values are explicit ratings.
2. **Book Lookup Table**: Curated metadata mapping `ISBN` to readable `Book-Title`, `Book-Author`, `Year-Of-Publication`, and `Publisher`.
3. **User-Level Aggregates**: Total rating count, average score, minimum rating, and maximum rating per user.
4. **Book-Level Aggregates**: Total rating count, average score, minimum rating, and maximum rating per book.


In [1]:
# 1. Book Metadata Lookup Table
book_lookup = books_silver[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher']].copy()
book_lookup.to_parquet('gold/book_lookup.parquet', index=False)

# 2. User-level Aggregates
user_aggregates = ratings_silver.groupby('User-ID')['Book-Rating'].agg(
    rating_count='count',
    avg_rating='mean',
    min_rating='min',
    max_rating='max'
).reset_index()
user_aggregates.to_parquet('gold/user_aggregates.parquet', index=False)

# 3. Book-level Aggregates
book_aggregates = ratings_silver.groupby('ISBN')['Book-Rating'].agg(
    rating_count='count',
    avg_rating='mean',
    min_rating='min',
    max_rating='max'
).reset_index()
book_aggregates = book_aggregates.merge(book_lookup[['ISBN', 'Book-Title', 'Book-Author']], on='ISBN', how='left')
book_aggregates.to_parquet('gold/book_aggregates.parquet', index=False)

# 4. Model-Ready Sparse Matrix (Books as Rows, Users as Columns)
unique_books = sorted(ratings_silver['ISBN'].unique())
unique_users = sorted(ratings_silver['User-ID'].unique())

book_to_idx = {isbn: idx for idx, isbn in enumerate(unique_books)}
idx_to_book = {idx: isbn for idx, isbn in enumerate(unique_books)}
user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
idx_to_user = {idx: uid for idx, uid in enumerate(unique_users)}

rows = ratings_silver['ISBN'].map(book_to_idx).values
cols = ratings_silver['User-ID'].map(user_to_idx).values
vals = ratings_silver['Book-Rating'].values

book_user_matrix = csr_matrix((vals, (rows, cols)), shape=(len(unique_books), len(unique_users)))

# Save matrix and index mapping
save_npz('gold/book_user_matrix.npz', book_user_matrix)
index_mapping = pd.DataFrame({'book_idx': list(idx_to_book.keys()), 'ISBN': list(idx_to_book.values())})
index_mapping.to_parquet('gold/book_index_mapping.parquet', index=False)

print("=== GOLD LAYER SUMMARY ===")
print(f"Book Lookup Shape        : {book_lookup.shape}")
print(f"User Aggregates Shape    : {user_aggregates.shape}")
print(f"Book Aggregates Shape    : {book_aggregates.shape}")
print(f"Sparse Matrix Dimensions : {book_user_matrix.shape} (Rows: Books, Cols: Users)")
print(f"Non-Zero Entries in CSR  : {book_user_matrix.nnz:,}")
print(f"Memory Footprint of CSR  : {book_user_matrix.data.nbytes / 1024 / 1024:.2f} MB")


=== GOLD LAYER SUMMARY ===
Book Lookup Shape        : (5440, 5)
User Aggregates Shape    : (6319, 5)
Book Aggregates Shape    : (5440, 7)
Sparse Matrix Dimensions : (5440, 6319) (Rows: Books, Cols: Users)
Non-Zero Entries in CSR  : 87,233
Memory Footprint of CSR  : 0.67 MB


---
## Step 5: Build the Recommender (Item-Based Collaborative Filtering)

### Methodological Foundation
In **Item-Based Collaborative Filtering**, each book $i$ is represented by an interaction vector $\vec{v}_i \in \mathbb{R}^M$ across all $M$ users.
The similarity between two books $i$ and $j$ is computed using the **Cosine Similarity**:

$$\text{Cosine Similarity}(i, j) = \cos(\vec{v}_i, \vec{v}_j) = \frac{\vec{v}_i \cdot \vec{v}_j}{\|\vec{v}_i\|_2 \|\vec{v}_j\|_2} = 1 - \text{cosine\_distance}(i, j)$$

### Personalized Recommendation Scoring
For a target user $u$:
1. Identify the books $I_u$ the user rated highly (ratings $\ge 7$).
2. For each highly rated book $i \in I_u$ with user rating $r_{u, i}$, find its nearest neighbors $j$ using the cosine model.
3. Exclude any books the user has already read.
4. Calculate the weighted recommendation score for unread candidate book $j$:

$$\text{Score}(u, j) = \sum_{i \in I_u} \text{sim}(i, j) \times r_{u, i}$$

5. Rank candidates in descending order of weighted score and return the top $n$ titles.


In [1]:
# 1. Load Gold Sparse Matrix and Metadata
sparse_matrix = load_npz('gold/book_user_matrix.npz')
book_lookup_df = pd.read_parquet('gold/book_lookup.parquet')
book_meta_dict = book_lookup_df.set_index('ISBN').to_dict('index')

# 2. Fit NearestNeighbors with Cosine Metric
model_nn = NearestNeighbors(metric='cosine', algorithm='brute')
model_nn.fit(sparse_matrix)

# 3. Function: get_similar_books(book_title, n=10)
def get_similar_books(book_title, n=10):
    '''Return the n most similar books to a given title using cosine similarity.'''
    matches = [isbn for isbn, meta in book_meta_dict.items() if book_title.lower() in meta['Book-Title'].lower()]
    if not matches:
        return f"Book title matching '{book_title}' not found in catalog."
    
    target_isbn = matches[0]
    target_idx = book_to_idx[target_isbn]
    
    distances, indices = model_nn.kneighbors(sparse_matrix[target_idx], n_neighbors=n+1)
    
    similar_books = []
    for d, idx in zip(distances[0][1:], indices[0][1:]):
        cand_isbn = idx_to_book[idx]
        info = book_meta_dict[cand_isbn]
        similar_books.append({
            'ISBN': cand_isbn,
            'Title': info['Book-Title'],
            'Author': info['Book-Author'],
            'Year': info['Year-Of-Publication'],
            'Cosine_Similarity': round(1.0 - d, 4)
        })
    return pd.DataFrame(similar_books)

# 4. Function: recommend(user_id, n=10, min_user_rating=7, sim_threshold=0.10)
def recommend(user_id, n=10, min_user_rating=7, sim_threshold=0.10):
    '''Generate personalized recommendations for a user based on rating-weighted item similarities.'''
    u_ratings = ratings_silver[ratings_silver['User-ID'] == user_id]
    if u_ratings.empty:
        return f"User {user_id} not found in model dataset."
    
    read_isbns = set(u_ratings['ISBN'].tolist())
    
    # Select user's highly rated books
    high_rated = u_ratings[u_ratings['Book-Rating'] >= min_user_rating]
    if len(high_rated) < 3:
        high_rated = u_ratings.sort_values(by='Book-Rating', ascending=False).head(5)
        
    candidate_scores = defaultdict(float)
    candidate_max_sim = defaultdict(float)
    
    for _, row in high_rated.iterrows():
        b_isbn = row['ISBN']
        b_rating = row['Book-Rating']
        if b_isbn not in book_to_idx:
            continue
        idx = book_to_idx[b_isbn]
        
        # Query 15 nearest neighbors for each favorite book
        distances, indices = model_nn.kneighbors(sparse_matrix[idx], n_neighbors=16)
        for d, n_idx in zip(distances[0][1:], indices[0][1:]):
            n_isbn = idx_to_book[n_idx]
            if n_isbn in read_isbns:
                continue
            sim = 1.0 - d
            if sim >= sim_threshold:
                candidate_scores[n_isbn] += sim * b_rating
                if sim > candidate_max_sim[n_isbn]:
                    candidate_max_sim[n_isbn] = sim
                    
    # Fallback to nearest neighbors if threshold filters out all candidates
    if not candidate_scores:
        for _, row in high_rated.head(5).iterrows():
            b_isbn = row['ISBN']
            b_rating = row['Book-Rating']
            if b_isbn not in book_to_idx:
                continue
            idx = book_to_idx[b_isbn]
            distances, indices = model_nn.kneighbors(sparse_matrix[idx], n_neighbors=6)
            for d, n_idx in zip(distances[0][1:], indices[0][1:]):
                n_isbn = idx_to_book[n_idx]
                if n_isbn not in read_isbns:
                    candidate_scores[n_isbn] += (1.0 - d) * b_rating
                    candidate_max_sim[n_isbn] = max(candidate_max_sim[n_isbn], 1.0 - d)

    ranked_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    
    recs = []
    for rank, (cand_isbn, score) in enumerate(ranked_candidates, 1):
        info = book_meta_dict.get(cand_isbn, {})
        recs.append({
            'Rank': rank,
            'ISBN': cand_isbn,
            'Title': info.get('Book-Title', 'Unknown'),
            'Author': info.get('Book-Author', 'Unknown'),
            'Year': info.get('Year-Of-Publication', 'Unknown'),
            'Weighted_Score': round(score, 3),
            'Max_Cosine_Sim': round(candidate_max_sim[cand_isbn], 4)
        })
    return pd.DataFrame(recs)

# Test title similarity function
print("=== Top 5 Similar Books to 'Harry Potter and the Chamber of Secrets' ===")
hp_similar = get_similar_books("Harry Potter and the Chamber of Secrets", n=5)
print(hp_similar[['Title', 'Author', 'Cosine_Similarity']].to_string(index=False))


=== Top 5 Similar Books to 'Harry Potter and the Chamber of Secrets' ===
                                                           Title        Author  Cosine_Similarity
               Harry Potter and the Prisoner of Azkaban (Book 3) J. K. Rowling             0.3893
Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)) J. K. Rowling             0.3699
                    Harry Potter and the Goblet of Fire (Book 4) J. K. Rowling             0.3362
              Harry Potter and the Order of the Phoenix (Book 5) J. K. Rowling             0.1797
                  Harry Potter and the Sorcerer's Stone (Book 1) J. K. Rowling             0.1376


---
## Step 6: Demo and Sanity Check

We test `recommend()` across **three distinct users** with different genre profiles:
1. **User 276964**: Epic Fantasy & Thrillers (*The Wheel of Time*, John Grisham, Stephen King, Tad Williams).
2. **User 278418**: Children's Literature & Romance (*Dr. Seuss*, Nora Roberts).
3. **User 254**: Modern Fantasy & Magical Realism (Neil Gaiman, *Harry Potter*, Sue Monk Kidd).

For each user, we print their highly rated books alongside the top 5 model recommendations.


In [1]:
demo_user_ids = [276964, 278418, 254]

for uid in demo_user_ids:
    print("=" * 80)
    print(f" DEMONSTRATION & SANITY CHECK FOR USER ID: {uid}")
    print("=" * 80)
    
    # 1. Print books user rated highly
    user_top_books = ratings_silver[ratings_silver['User-ID'] == uid].sort_values(by='Book-Rating', ascending=False).head(5)
    print("\n[Books the User Rated Highly]:")
    for _, r in user_top_books.iterrows():
        b = book_meta_dict.get(r['ISBN'], {})
        print(f"  * Rating {r['Book-Rating']:2d}/10 : \"{b.get('Book-Title')}\" by {b.get('Book-Author')}")
        
    # 2. Print recommendations
    print("\n[Top Recommended Books for this User]:")
    recs_df = recommend(uid, n=5)
    print(recs_df[['Rank', 'Title', 'Author', 'Weighted_Score', 'Max_Cosine_Sim']].to_string(index=False))
    print()


 DEMONSTRATION & SANITY CHECK FOR USER ID: 276964

[Books the User Rated Highly]:
  * Rating 10/10 : "The Terminal Man" by Michael Crichton
  * Rating 10/10 : "The Fires of Heaven (The Wheel of Time, Book 5)" by Robert Jordan
  * Rating  9/10 : "The Chamber" by John Grisham
  * Rating  8/10 : "The Girl Who Loved Tom Gordon : A Novel" by Stephen King
  * Rating  8/10 : "Stone of Farewell (Memory, Sorrow, and Thorn, Book 2)" by Tad Williams

[Top Recommended Books for this User]:
 Rank                                                               Title        Author  Weighted_Score  Max_Cosine_Sim
    1                       The Shadow Rising (The Wheel of Time, Book 4) Robert Jordan          11.028          0.6340
    2    To Green Angel Tower, Part 1 (Memory, Sorrow, and Thorn, Book 3)  Tad Williams           9.302          0.7922
    3                           Lord of Chaos (The Wheel of Time, Book 6) Robert Jordan           8.933          0.6229
    4 Dragonbone Chair Sorrow 1 (Memo

### Qualitative Analysis & Sanity Verification

The demo results confirm the accuracy and personalization of our Item-Based Collaborative Filtering engine:

1. **User 276964 (Epic Fantasy Fan):**
   - **Input:** Highly rated *The Fires of Heaven (Wheel of Time 5)* and *Stone of Farewell (Memory, Sorrow & Thorn 2)*.
   - **Output:** The system accurately recommended *The Shadow Rising (Wheel of Time 4)* and *Lord of Chaos (Wheel of Time 6)* by Robert Jordan, alongside *To Green Angel Tower* and *Dragonbone Chair* by Tad Williams. The model successfully recognized multi-volume series continuity and author style without any hardcoded author metadata!

2. **User 278418 (Children & Romance Fan):**
   - **Input:** Highly rated *Green Eggs and Ham* by Dr. Seuss and *Dance upon the Air* by Nora Roberts.
   - **Output:** The model recommended *Fox in Socks* (Dr. Seuss) and *Heaven and Earth (Three Sisters Island Trilogy Book 2)* by Nora Roberts. The recommender captured both distinct reading interests independently.

3. **User 254 (Fantasy & Magical Realism Fan):**
   - **Input:** Highly rated Neil Gaiman and *Harry Potter and the Chamber of Secrets*.
   - **Output:** The engine returned the remaining installments of *Harry Potter* (*Order of the Phoenix*, *Goblet of Fire*, *Sorcerer's Stone*) and Tolkien's *The Return of the King*. Unread books were successfully filtered out, and the user received high-confidence, genre-aligned suggestions.

---

## Conclusion & Medallion Data Lake Benefits

- **Traceability & Lineage:** The **Bronze layer** guarantees that the raw data remains untouched and reproducible, with timestamps tracking exact ingestion instances.
- **Data Quality & Density:** The **Silver layer** resolved missing values, stripped invalid age/year values, and condensed matrix sparsity from **99.9968% down to 99.7462%** (a 79x density gain), transforming an untrainable dataset into an actionable interaction graph.
- **Fast Model Inference:** The **Gold layer** provides compressed CSR sparse matrices and indexed lookup tables, allowing nearest-neighbor inference in milliseconds with minimal RAM overhead (< 1 MB).
- **Personalized Recommendations:** Item-based collaborative filtering effectively balances series continuity and serendipitous discovery while eliminating items already read by the user.
